In [1]:
import pandas as pd


data = pd.read_csv("./data/email_spam_dataset.csv")

data


,email_text,label
0,Here is the project update you asked for.,ham
1,Limited offer!!! Buy now and get 50% discount.,spam
2,Win cash prizes instantly by replying to this ...,spam
3,Urgent! Your account has been suspended. Verif...,spam
4,"Hi, please find the meeting agenda attached.",ham
...,...,...
315,Here is the project update you asked for.,ham
316,You are selected for a free gift card. Act fast.,spam
317,Win cash prizes instantly by replying to this ...,spam
318,Limited offer!!! Buy now and get 50% discount.,spam


                     ### clearing data

In [2]:


data['label'].value_counts()
data['label'] = data["label"].map({
    "ham":0,
    "spam":1
})
data.head()

,email_text,label
0,Here is the project update you asked for.,0
1,Limited offer!!! Buy now and get 50% discount.,1
2,Win cash prizes instantly by replying to this ...,1
3,Urgent! Your account has been suspended. Verif...,1
4,"Hi, please find the meeting agenda attached.",0


###                    MODEL CREATing By Using Ann

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer


X = data["email_text"]
y = data['label']


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


vetorization = TfidfVectorizer()


X_train_tfidf = vetorization.fit_transform(X_train)
X_test_tfidf = vetorization.transform(X_test)


X_train_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2260 stored elements and shape (256, 71)>

In [4]:
import torch


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader


In [ ]:

class EmailData(Dataset):

    def __init__(self,features,label):
        self.features = features
        self.label = label


    def __len__(self):
        return self.features.shape[0]


    def __getitem__(self, index):
        return self.features[index],self.label[index]
       
        





X_train_tensor = torch.tensor(
    X_train_tfidf.toarray(),
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_tfidf.toarray(),
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train.values,
    dtype=torch.float32
).reshape(-1, 1)

y_test_tensor = torch.tensor(
    y_test.values,
    dtype=torch.float32
).reshape(-1, 1)

train_data = EmailData(X_train_tensor,y_train_tensor)

input_size = X_train_tensor.shape[1]


71

In [ ]:

class Models(nn.Module):

    def __init__(self,x):
        super().__init__()
        self.network = nn.Sequential(

            nn.Linear(x,128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,1)
          


        )
    def forward(self,x):
        out = self.network(x)
        return out

     


model = Models(input_size)

loss_function = nn.BCEWithLogitsLoss()

optimezer = optim.Adam(
     model.parameters(),
     lr=0.001
)

training_Data = DataLoader(train_data,batch_size=64,shuffle=True)



In [22]:


epochs = 100
print("Before training:", model(X_train_tensor[:5]))

for epoch in range(epochs):
    for features,label in training_Data:

        model.train()
        output = model(features)
    
        loss = loss_function(output, label)

        optimezer.zero_grad()
        loss.backward()
        optimezer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

print("After training:", model(X_train_tensor[:5]))

    

      



Before training: tensor([[ 11.9968],
        [-10.0031],
        [ 11.9461],
        [-10.0031],
        [ 11.2835]], grad_fn=<AddmmBackward0>)
Epoch [10/100], Loss: 0.0000
Epoch [20/100], Loss: 0.0000
Epoch [30/100], Loss: 0.0000
Epoch [40/100], Loss: 0.0000
Epoch [50/100], Loss: 0.0000
Epoch [60/100], Loss: 0.0000
Epoch [70/100], Loss: 0.0000
Epoch [80/100], Loss: 0.0000
Epoch [90/100], Loss: 0.0000
Epoch [100/100], Loss: 0.0000
After training: tensor([[ 13.7280],
        [-11.8671],
        [ 13.7625],
        [-11.8671],
        [ 13.0779]], grad_fn=<AddmmBackward0>)


In [23]:

test_data = EmailData(X_test_tensor,y_test_tensor)
print(len(test_data))
testing_dataLoader = DataLoader(test_data,batch_size=20,shuffle=True)

model.eval()

with torch.no_grad():
    outputs = model(X_test_tensor)

probabilities = torch.sigmoid(outputs)


prediction = (probabilities >= 0.5).float()
prediction




64


tensor([[1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [1.],
        [0.]])

In [ ]:
torch.save(model.state_dict(), "spam_classifier.pth")


In [ ]:
for data in range(0,100,10):
    print(data)